In [ ]:
# Install dependencies
!pip install kaggle facenet-pytorch torchvision


: 

In [ ]:
# Upload kaggle.json
from google.colab import files
files.upload()

# Setup Kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download dataset
!kaggle datasets download -d manjilkarki/deepfake-and-real-images
!unzip deepfake-and-real-images.zip

In [ ]:
!pip install mtcnn

In [ ]:
import os
import cv2
from mtcnn import MTCNN
from tqdm import tqdm

# Paths
train_real = "Dataset/Train/Real"
train_fake = "Dataset/Train/Fake"

processed_real = "/content/processed_dataset/real"
processed_fake = "/content/processed_dataset/fake"

os.makedirs(processed_real, exist_ok=True)
os.makedirs(processed_fake, exist_ok=True)

detector = MTCNN()

def process_folder(input_path, output_path, limit=12500):
    for img_name in tqdm(sorted(os.listdir(input_path))[:limit]):
        img_path = os.path.join(input_path, img_name)

        try:
            img = cv2.imread(img_path)
            if img is None:
                continue

            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            results = detector.detect_faces(img_rgb)

            if len(results) > 0:
                x, y, w, h = results[0]['box']
                x, y = max(0, x), max(0, y)
                face = img[y:y+h, x:x+w]

                if face.size == 0:
                    continue
            else:
                face = img

            face = cv2.resize(face, (224,224))
            save_path = os.path.join(output_path, img_name)

            cv2.imwrite(save_path, face)

        except:
            continue

# Process (LIMITED DATASET)
process_folder(train_real, processed_real, limit=12500)
process_folder(train_fake, processed_fake, limit=12500)

print("Preprocessing Done")

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# Transform (VERY IMPORTANT)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Load dataset
dataset = datasets.ImageFolder("/content/processed_dataset", transform=transform)

# Split (80% train, 20% validation)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_data, val_data = random_split(dataset, [train_size, val_size])

# DataLoaders
train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
val_loader = DataLoader(val_data, batch_size=16)

print("Dataset Ready")

In [ ]:
# Model Training

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import models
from tqdm import tqdm

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    auc
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# MODEL SETUP
# -----------------------------
model = models.resnet18(weights='DEFAULT')
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# -----------------------------
# TRACKING
# -----------------------------
epochs = 5
train_losses = []
val_accuracies = []

# -----------------------------
# TRAINING LOOP
# -----------------------------
for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in tqdm(train_loader):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    train_losses.append(total_loss)

    # -----------------------------
    # VALIDATION
    # -----------------------------
    model.eval()
    correct = 0
    total = 0

    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 0].cpu().numpy())  # probability of FAKE (class 0)

    accuracy = 100 * correct / total
    val_accuracies.append(accuracy)

    print(f"Epoch {epoch+1}")
    print(f"Loss: {total_loss:.4f}, Val Accuracy: {accuracy:.2f}%")

# -----------------------------
# CONFUSION MATRIX
# -----------------------------
cm = confusion_matrix(all_labels, all_preds)
print("\nConfusion Matrix:\n", cm)

plt.figure()
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.colorbar()

plt.xlabel("Predicted")
plt.ylabel("Actual")

for i in range(len(cm)):
    for j in range(len(cm)):
        plt.text(j, i, cm[i][j], ha='center', va='center')

plt.show()

# -----------------------------
# METRICS
# -----------------------------
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

print("\nPrecision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

# -----------------------------
# ROC CURVE
# -----------------------------
fpr, tpr, _ = roc_curve(all_labels, all_probs)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC Curve (AUC = {roc_auc:.2f})")
plt.show()

# -----------------------------
# ACCURACY GRAPH
# -----------------------------
plt.figure()
plt.plot(range(1, epochs+1), val_accuracies)
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy")
plt.title("Accuracy vs Epoch")
plt.show()

# -----------------------------
# LOSS GRAPH
# -----------------------------
plt.figure()
plt.plot(range(1, epochs+1), train_losses)
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Loss vs Epoch")
plt.show()

# -----------------------------
# SAVE MODEL
# -----------------------------
torch.save(model.state_dict(), "/content/resnet18_deepfake.pth")
print("Model Saved")

In [ ]:
# Save trained model permanently on Drive
import os

os.makedirs("/content/drive/MyDrive/deepfake_model", exist_ok=True)

torch.save(model.state_dict(),
           "/content/drive/MyDrive/deepfake_models/resnet18_deepfake_model.pth")

In [ ]:
# Save trained model locally on the system.
from google.colab import files
files.download("/content/drive/MyDrive/deepfake_models/resnet18_deepfake.pth")

In [ ]:
# ===========================================================
#                     GradCAM Integration
# ===========================================================

In [ ]:
# STEP 1 — INSTALL LIBRARY
!pip install grad-cam

In [ ]:
# STEP 2 — IMPORTS
import cv2
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from torchvision import transforms

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

In [ ]:
# STEP 3 — LOAD TRAINED MODEL

import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 2)

model.load_state_dict(
    torch.load("/content/drive/MyDrive/deepfake_models/resnet18_deepfake_model.pth")
)

model = model.to(device)
model.eval()

In [ ]:
# STEP 4 — IMAGE PREPROCESSING

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

image_path = "/content/aish.jpg"

pil_image = Image.open(image_path).convert("RGB")

input_tensor = transform(pil_image).unsqueeze(0).to(device)

rgb_image = np.array(pil_image.resize((224, 224))) / 255.0

# STEP 5 — PREDICTION

outputs = model(input_tensor)

probabilities = torch.softmax(outputs, dim=1)

predicted_class = torch.argmax(probabilities).item()

classes = ["Real", "Fake"]

print("Prediction:", classes[predicted_class])

print("Confidence:", probabilities[0][predicted_class].item())

# STEP 6 — GRAD-CAM

target_layers = [model.layer4[-1]]

cam = GradCAM(
    model=model,
    target_layers=target_layers
)

targets = [ClassifierOutputTarget(predicted_class)]

grayscale_cam = cam(
    input_tensor=input_tensor,
    targets=targets
)

grayscale_cam = grayscale_cam[0]


In [ ]:
# STEP 7 — OVERLAY HEATMAP

visualization = show_cam_on_image(
    rgb_image,
    grayscale_cam,
    use_rgb=True
)

plt.figure(figsize=(8,8))
plt.imshow(visualization)
plt.title("Grad-CAM Visualization")
plt.axis('off')
plt.show()

In [ ]:
# VQGAN Integration

In [ ]:
# Step 1: Install Dependencies

!git clone https://github.com/CompVis/taming-transformers.git
%cd taming-transformers

!pip install pytorch-lightning
!pip install omegaconf
!pip install einops

In [ ]:
import os

print(os.getcwd())
!ls

!find /content -name "*.ckpt"

!find /content -name "*.yaml"

In [ ]:
# Step 2: Install the Repository

!mkdir -p checkpoints
%cd checkpoints

!wget https://heibox.uni-heidelberg.de/f/140747ba53464f49b476/?dl=1 -O imagenet_vqgan.ckpt

In [ ]:
!ls -lh

!ls -lh checkpoints

In [ ]:
%cd /content/taming-transformers

In [ ]:
!pip install -e .

In [ ]:
# Step 3: Restart Runtime Session (Important)

file_path = "/content/taming-transformers/taming/data/utils.py"

with open(file_path, "r") as f:
    content = f.read()

content = content.replace(
    "from torch._six import string_classes",
    "string_classes = (str,)"
)

with open(file_path, "w") as f:
    f.write(content)

print("Patch Applied")

In [ ]:
# Step 4: Verify

%cd /content/taming-transformers

from taming.models.vqgan import VQModel
from omegaconf import OmegaConf

print("Imports Successful")

In [ ]:
config = OmegaConf.load(
    "/content/taming-transformers/configs/imagenet_vqgan.yaml"
)

print(config.model.target)

print(config.model.params.keys())

In [ ]:
# Step 5: Create the VQGAN Model Object

model = VQModel(**config.model.params)

print(type(model))

In [ ]:
# Step 6: Load the Checkpoint
import torch

checkpoint = torch.load(
    "/content/taming-transformers/checkpoints/imagenet_vqgan.ckpt",
    map_location="cpu"
)

print(checkpoint.keys())

for key in checkpoint.keys():
    print(key)

In [ ]:
# Step 7: Load the Weights into VQGAN

missing, unexpected = model.load_state_dict(
    checkpoint["state_dict"],
    strict=False
)

print("Missing Keys:", len(missing))
print("Unexpected Keys:", len(unexpected))

In [ ]:
# Step 8: Move Model to GPU

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)
model.eval()

print("Model Ready on:", device)


In [ ]:
print(hasattr(model, "encode"))
print(hasattr(model, "decode"))

In [ ]:
# Step 10: Load a Test Image

In [ ]:
# Step 10: Load a Test Image

from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt

image_path = "/content/Arpita.jpeg"

img = Image.open(image_path).convert("RGB")

transform = transforms.Compose([
    transforms.Resize((256,256)),
    transforms.ToTensor()
])

img_tensor = transform(img).unsqueeze(0).to(device)

print(img_tensor.shape)

In [ ]:
# Step 11: Encode

with torch.no_grad():
    z = model.encode(img_tensor)

print("Length:", len(z))

for i, item in enumerate(z):
    print(f"\nItem {i}:")
    print(type(item))

    if hasattr(item, "shape"):
        print("Shape:", item.shape)


In [ ]:
with torch.no_grad():
    quant = z[0]

    reconstruction = model.decode(quant)

print(reconstruction.shape)

In [ ]:
import matplotlib.pyplot as plt

recon_img = reconstruction.squeeze(0).cpu()

recon_img = recon_img.permute(1, 2, 0)

recon_img = recon_img.numpy()

recon_img = (recon_img + 1) / 2

recon_img = recon_img.clip(0, 1)

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.imshow(img)
plt.title("Original")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(recon_img)
plt.title("VQGAN Reconstruction")
plt.axis("off")

plt.show()